In [1]:
import json
import torch
from datasets import load_dataset, DatasetDict

# --- 1. Global Configuration ---
DATA_FILES = {
    'train': 'dataset/train.json',
    'validation': 'dataset/valid.json',
    'test': 'dataset/test.json'
}
LABEL_FILE = 'dataset/label.json'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {device}")

# --- 2. Load Raw Data ---
print("⏳ Loading dataset from local files...")
raw_datasets = load_dataset('json', data_files=DATA_FILES)
print("✅ Raw dataset loaded successfully!")

# --- 3. Load & Define Master Label Mappings ---
with open(LABEL_FILE, 'r') as f:
    label_list = json.load(f)

ID2LABEL = {str(i): label for i, label in enumerate(label_list)}
LABEL2ID = {label: str(i) for i, label in enumerate(label_list)}
print("\n✅ Master label mappings established:")
print(f"Total Labels: {len(label_list)}")
print(f"ID2LABEL: {ID2LABEL}")

s:\Masters-EDS_Fall25\NLP\Final_Project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Using device: cuda
⏳ Loading dataset from local files...
✅ Raw dataset loaded successfully!

✅ Master label mappings established:
Total Labels: 5
ID2LABEL: {'0': 'O', '1': 'B-Chemical', '2': 'B-Disease', '3': 'I-Disease', '4': 'I-Chemical'}


In [2]:
import transformers
from transformers import AutoTokenizer

# --- 1. Track A Configuration ---
MODEL_CHECKPOINT_A = "dmis-lab/biobert-base-cased-v1.1"

print(f"⏳ Loading tokenizer for {MODEL_CHECKPOINT_A}...")
TOKENIZER_A = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT_A)
print("✅ BioBERT Tokenizer loaded.")

# --- 2. Alignment Function ---
def align_labels(examples, tokenizer_obj, id2label_map, label2id_map):
    tokenized_inputs = tokenizer_obj(
        examples["tokens"], 
        truncation=True, 
        is_split_into_words=True
    )

    labels = []
    for i, label_ids in enumerate(examples["tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        new_label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                new_label_ids.append(-100)
            elif word_idx != previous_word_idx:
                new_label_ids.append(label_ids[word_idx])
            else:
                original_tag_id = label_ids[word_idx]
                label_name = id2label_map.get(str(original_tag_id))
                if label_name and label_name.startswith('B-'):
                    i_label_name = 'I-' + label_name[2:]
                    new_label_ids.append(int(label2id_map.get(i_label_name, original_tag_id)))
                else:
                    new_label_ids.append(original_tag_id)
            previous_word_idx = word_idx
        labels.append(new_label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# --- 3. Execute Preprocessing for Track A ---
print("\n⏳ Starting Token-Label Alignment (Track A)...")
tokenized_datasets_A = raw_datasets.map(
    align_labels,
    batched=True,
    num_proc=1,
    load_from_cache_file=False,
    desc="Aligning labels for BioBERT",
    fn_kwargs={"tokenizer_obj": TOKENIZER_A, "id2label_map": ID2LABEL, "label2id_map": LABEL2ID}
)

# --- 4. Add word_ids to test set (for advanced evaluation) ---
def add_word_ids(example, tokenizer_obj):
    word_ids = tokenizer_obj(example["tokens"], truncation=True, is_split_into_words=True).word_ids()
    return {"word_ids": word_ids}

print("⏳ Adding word_ids to Track A test set...")
tokenized_datasets_A["test"] = tokenized_datasets_A["test"].map(
    add_word_ids, 
    fn_kwargs={"tokenizer_obj": TOKENIZER_A},
    num_proc=1
)
print("✅ Track A Data is ready!")

⏳ Loading tokenizer for dmis-lab/biobert-base-cased-v1.1...
✅ BioBERT Tokenizer loaded.

⏳ Starting Token-Label Alignment (Track A)...


Aligning labels for BioBERT (num_proc=1): 100%|██████████| 5865/5865 [00:27<00:00, 216.15 examples/s]


⏳ Adding word_ids to Track A test set...


Map (num_proc=1): 100%|██████████| 5865/5865 [00:14<00:00, 395.04 examples/s] 

✅ Track A Data is ready!


In [3]:
import numpy as np
import evaluate
from transformers import (
    DataCollatorForTokenClassification, 
    AutoModelForTokenClassification, 
    TrainingArguments, 
    Trainer,
    EarlyStoppingCallback
)

# --- 1. Metrics Setup ---
METRIC = evaluate.load("seqeval")

def compute_metrics_transformer(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [ID2LABEL[str(p)] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [ID2LABEL[str(l)] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = METRIC.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

# --- 2. Model A Loading ---
print(f"⏳ Loading BioBERT Model: {MODEL_CHECKPOINT_A}...")
model_A = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT_A,
    num_labels=len(ID2LABEL),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    weights_only=False # Safety bypass for old PyTorch
)

# --- 3. Trainer A Configuration ---
args_A = TrainingArguments(
    output_dir="biobert_track_a_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
    fp16=torch.cuda.is_available(),
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
)

trainer_A = Trainer(
    model=model_A,
    args=args_A,
    train_dataset=tokenized_datasets_A["train"],
    eval_dataset=tokenized_datasets_A["validation"],
    tokenizer=TOKENIZER_A,
    data_collator=DataCollatorForTokenClassification(tokenizer=TOKENIZER_A),
    compute_metrics=compute_metrics_transformer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("\n🚀 Starting Track A (BioBERT) training...")
trainer_A.train()

⏳ Loading BioBERT Model: dmis-lab/biobert-base-cased-v1.1...


Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
3239411603.py:62: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_A = Trainer(



🚀 Starting Track A (BioBERT) training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,0.115473,0.805889,0.833281,0.819356
2,0.347700,0.110618,0.841930,0.871338,0.856382
3,0.347700,0.119135,0.849404,0.884475,0.866585
4,0.050800,0.135306,0.857975,0.891252,0.874297
5,0.018600,0.155802,0.855767,0.889584,0.872348
6,0.018600,0.166396,0.875724,0.882390,0.879044
7,0.009900,0.182211,0.870011,0.893233,0.881469
8,0.004600,0.200123,0.863121,0.896778,0.879628
9,0.004600,0.201584,0.866700,0.894172,0.880222
10,0.001800,0.201060,0.867433,0.896465,0.881711


TrainOutput(global_step=3270, training_loss=0.06639472110373529, metrics={'train_runtime': 1023.0894, 'train_samples_per_second': 51.1, 'train_steps_per_second': 3.196, 'total_flos': 2176591610353680.0, 'train_loss': 0.06639472110373529, 'epoch': 10.0})

In [ ]:
# --- 1. Track B Configuration ---
MODEL_CHECKPOINT_B = "microsoft/deberta-v3-base"

print(f"⏳ Loading tokenizer for {MODEL_CHECKPOINT_B}...")
TOKENIZER_B = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT_B)
print("✅ DeBERTa Tokenizer loaded.")

# --- 2. Execute Preprocessing for Track B ---
print("\n⏳ Starting Token-Label Alignment (Track B)...")
tokenized_datasets_B = raw_datasets.map(
    align_labels,
    batched=True,
    num_proc=1,
    load_from_cache_file=False,
    desc="Aligning labels for DeBERTa",
    fn_kwargs={"tokenizer_obj": TOKENIZER_B, "id2label_map": ID2LABEL, "label2id_map": LABEL2ID}
)

# --- 3. Add word_ids to test set (for advanced evaluation) ---
print("⏳ Adding word_ids to Track B test set...")
tokenized_datasets_B["test"] = tokenized_datasets_B["test"].map(
    add_word_ids, 
    fn_kwargs={"tokenizer_obj": TOKENIZER_B},
    num_proc=1
)
print("✅ Track B Data is ready!")

# --- 4. Model B Loading ---
print(f"\n⏳ Loading DeBERTa Model: {MODEL_CHECKPOINT_B}...")
model_B = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT_B,
    num_labels=len(ID2LABEL),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    weights_only=False
)

# --- 5. Trainer B Configuration ---
args_B = TrainingArguments(
    output_dir="deberta_track_b_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
    fp16=torch.cuda.is_available(),
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
)

trainer_B = Trainer(
    model=model_B,
    args=args_B,
    train_dataset=tokenized_datasets_B["train"],
    eval_dataset=tokenized_datasets_B["validation"],
    tokenizer=TOKENIZER_B,
    data_collator=DataCollatorForTokenClassification(tokenizer=TOKENIZER_B),
    compute_metrics=compute_metrics_transformer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("\n🚀 Starting Track B (DeBERTa) training...")
trainer_B.train()

⏳ Loading tokenizer for microsoft/deberta-v3-base...


s:\Masters-EDS_Fall25\NLP\Final_Project\.venv\lib\site-packages\transformers\convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


✅ DeBERTa Tokenizer loaded.

⏳ Starting Token-Label Alignment (Track B)...


Aligning labels for DeBERTa (num_proc=1): 100%|██████████| 5865/5865 [00:20<00:00, 279.44 examples/s]


⏳ Adding word_ids to Track B test set...


Map (num_proc=1): 100%|██████████| 5865/5865 [00:24<00:00, 240.93 examples/s] 


... (2 outputs left out)


🚀 Starting Track B (DeBERTa) training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,0.099488,0.806899,0.873110,0.838700


KeyboardInterrupt: 

: 

In [ ]:
import os
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from gensim.models import KeyedVectors

# --- 1. Build Vocabulary (FIXED: all lowercase) ---
print("⏳ Building lowercase vocabulary...")
word_counts = Counter()
for example in raw_datasets['train']:
    word_counts.update([w.lower() for w in example['tokens']])

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
word2id = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for word, _ in word_counts.most_common():
    word2id[word] = len(word2id)
id2word = {v: k for k, v in word2id.items()}
VOCAB_SIZE = len(word2id)
print(f"✅ Lowercase vocabulary built! Size: {VOCAB_SIZE}")

# --- 2. Define Dataset Class (FIXED: all lowercase) ---
class NERDataset(Dataset):
    def __init__(self, hf_dataset, word2id):
        self.dataset = hf_dataset
        self.word2id = word2id
        self.unk_id = word2id["<UNK>"]

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        # Convert words to lowercase IDs
        token_ids = [self.word2id.get(w.lower(), self.unk_id) for w in item['tokens']]
        label_ids = item['tags']
        return {
            'token_ids': torch.tensor(token_ids, dtype=torch.long),
            'label_ids': torch.tensor(label_ids, dtype=torch.long)
        }

# --- 3. Define Collate Function ---
def collate_fn(batch):
    token_seqs = [item['token_ids'] for item in batch]
    label_seqs = [item['label_ids'] for item in batch]
    padded_tokens = pad_sequence(token_seqs, batch_first=True, padding_value=0)
    padded_labels = pad_sequence(label_seqs, batch_first=True, padding_value=-100) # -100 is fine here
    attention_masks = (padded_tokens != 0).long()
    return padded_tokens, padded_labels, attention_masks

# --- 4. Load Embeddings (Gensim) ---
def load_embeddings_gensim(path, word2id, embedding_dim, binary=False):
    print(f"⏳ Loading embeddings from {path}...")
    vocab_size = len(word2id)
    embedding_matrix = np.random.uniform(-0.25, 0.25, (vocab_size, embedding_dim))
    embedding_matrix[word2id["<PAD>"]] = np.zeros((embedding_dim,))
    
    try:
        kv = KeyedVectors.load_word2vec_format(path, binary=binary, limit=500000)
        hit_count = 0
        for word, idx in word2id.items():
            if word in kv:
                embedding_matrix[idx] = kv[word]
                hit_count += 1
        print(f"✅ Loaded! Coverage (Hit Rate): {hit_count / vocab_size:.2%}")
        return embedding_matrix
    except FileNotFoundError:
        print(f"❌ ERROR: File not found at {path}. Make sure it's in the './embeddings' folder.")
        return None

print("\n--- Loading Embeddings (using new lowercase vocab) ---")
embedding_matrix_fasttext = load_embeddings_gensim(
    "./embeddings/wiki-news-300d-1M.vec", word2id, 300, binary=False
)
embedding_matrix_bio = load_embeddings_gensim(
    "./embeddings/bio_embedding_intrinsic", word2id, 200, binary=True
)

# --- 5. Create DataLoaders ---
BATCH_SIZE = 32
train_dataset = NERDataset(raw_datasets['train'], word2id)
valid_dataset = NERDataset(raw_datasets['validation'], word2id)
test_dataset = NERDataset(raw_datasets['test'], word2id)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print("\n✅ Track C DataLoaders are ready!")

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torchcrf import CRF
from tqdm import tqdm

# --- 1. Define Model Architecture ---
class BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_labels, embedding_matrix=None):
        super(BiLSTM_CRF, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        if embedding_matrix is not None:
            print(f"🔧 Loading pre-trained {embedding_dim}d weights...")
            self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))
            # self.embedding.weight.requires_grad = False # Optional: Freeze embeddings
        
        self.lstm = nn.LSTM(embedding_dim, hidden_dim // 2, 
                            num_layers=1, bidirectional=True, batch_first=True)
        self.hidden2tag = nn.Linear(hidden_dim, num_labels)
        self.crf = CRF(num_labels, batch_first=True)
        
    def forward(self, input_ids, mask):
        embeds = self.embedding(input_ids)
        lstm_out, _ = self.lstm(embeds)
        emissions = self.hidden2tag(lstm_out)
        return emissions

    def compute_loss(self, emissions, tags, mask):
        return -self.crf(emissions, tags, mask=mask.bool())

    def decode(self, emissions, mask):
        return self.crf.decode(emissions, mask=mask.bool())

# --- 2. Define Enhanced Training Loop (Returns History) ---
def train_bilstm(model, train_loader, valid_loader, epochs=10, learning_rate=0.01, patience=3, name="Model"):
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    best_f1 = -1.0  # so epoch 1 always saves a checkpoint
    epochs_no_improve = 0
    history = {'train_loss': [], 'val_f1': []} # For learning curves
    
    print(f"\n🚀 Starting training for {name}...")
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            tokens, labels, mask = [b.to(device) for b in batch]
            safe_labels = torch.where(mask.bool(), labels, torch.tensor(0, device=device))
            model.zero_grad()
            emissions = model(tokens, mask)
            loss = model.compute_loss(emissions, safe_labels, mask)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
        
        avg_train_loss = total_loss / len(train_loader)
        history['train_loss'].append(avg_train_loss)
        
        model.eval()
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for batch in valid_loader:
                tokens, labels, mask = [b.to(device) for b in batch]
                emissions = model(tokens, mask)
                pred_tags = model.decode(emissions, mask)
                
                for i, sent_preds in enumerate(pred_tags):
                    real_len = mask[i].sum().item()
                    sent_labels = labels[i][:real_len].cpu().numpy()
                    sent_preds = sent_preds[:real_len]
                    all_preds.append([ID2LABEL[str(p)] for p in sent_preds])
                    all_labels.append([ID2LABEL[str(l)] for l in sent_labels])

        results = METRIC.compute(predictions=all_preds, references=all_labels)
        f1 = results["overall_f1"]
        history['val_f1'].append(f1)
        
        print(f"📊 Epoch {epoch+1}: Train Loss={avg_train_loss:.4f} | Val F1={f1:.4%}")
        
        if f1 > best_f1:
            best_f1 = f1
            epochs_no_improve = 0
            torch.save(model.state_dict(), f"{name}_best.pth")
            print("🌟 New best model saved!")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"\n🛑 Early stopping triggered at epoch {epoch+1}.")
                break

    print(f"\n🏁 Training complete for {name}! Best Val F1: {best_f1:.4%}")
    model.load_state_dict(torch.load(f"{name}_best.pth"))
    return model, history

# --- 3. Instantiate & Train Track C Models ---
HIDDEN_DIM = 256
NUM_LABELS = len(LABEL2ID)

# --- Track C1: BioWordVec ---
model_c1 = BiLSTM_CRF(VOCAB_SIZE, 200, HIDDEN_DIM, NUM_LABELS, embedding_matrix_bio).to(device)
model_c1_trained, history_c1 = train_bilstm(model_c1, train_loader, valid_loader, name="Track_C1_BioWordVec")

# --- Track C2: FastText ---
model_c2 = BiLSTM_CRF(VOCAB_SIZE, 300, HIDDEN_DIM, NUM_LABELS, embedding_matrix_fasttext).to(device)
model_c2_trained, history_c2 = train_bilstm(model_c2, train_loader, valid_loader, name="Track_C2_FastText")

In [ ]:
from seqeval.metrics import classification_report, f1_score
from collections import Counter
import pandas as pd

# --- Function for Transformers (Strategy 2b: Prioritized Vote) ---
def evaluate_transformer(trainer, test_dataset):
    print(f"\n⏳ Evaluating {trainer.model.name_or_path} (Strategy 2b: Prioritized Vote)...")
    
    predictions_output, label_ids, _ = trainer.predict(test_dataset)
    predictions = np.argmax(predictions_output, axis=2)
    all_word_ids = test_dataset['word_ids']

    reconciled_preds, reconciled_labels = [], []

    for i in range(len(all_word_ids)):
        word_ids, preds, labels = all_word_ids[i], predictions[i], label_ids[i]
        reconciled_sentence_preds, reconciled_sentence_labels = [], []
        current_word_idx, current_word_preds, current_word_true_label = None, set(), 'O'
        
        for j, word_idx in enumerate(word_ids):
            if word_idx != current_word_idx:
                if current_word_idx is not None:
                    winner_label = 'O'
                    b_tags = [p for p in current_word_preds if p.startswith('B-')]
                    if b_tags: winner_label = b_tags[0]
                    else:
                        i_tags = [p for p in current_word_preds if p.startswith('I-')]
                        if i_tags: winner_label = i_tags[0]
                    reconciled_sentence_preds.append(winner_label)
                    reconciled_sentence_labels.append(current_word_true_label)
                
                current_word_preds, current_word_idx = set(), word_idx
                if word_idx is not None:
                    current_word_true_label = ID2LABEL[str(labels[j])]
                    current_word_preds.add(ID2LABEL[str(preds[j])])
                else: current_word_true_label = 'O'
            elif word_idx is not None:
                current_word_preds.add(ID2LABEL[str(preds[j])])
        
        if current_word_idx is not None:
            winner_label = 'O'
            b_tags = [p for p in current_word_preds if p.startswith('B-')]
            if b_tags: winner_label = b_tags[0]
            else:
                i_tags = [p for p in current_word_preds if p.startswith('I-')]
                if i_tags: winner_label = i_tags[0]
            reconciled_sentence_preds.append(winner_label)
            reconciled_sentence_labels.append(current_word_true_label)

        reconciled_preds.append(reconciled_sentence_preds)
        reconciled_labels.append(reconciled_sentence_labels)

    report_str = classification_report(reconciled_labels, reconciled_preds, digits=4)
    report_dict = classification_report(reconciled_labels, reconciled_preds, output_dict=True)
    print("✅ Prioritized vote reconciliation complete. Final Results:")
    print(report_str)
    return report_dict, reconciled_labels, reconciled_preds

# --- Function for BiLSTM-CRF (FIXED torch.load) ---
def evaluate_bilstm(model_path, model_instance, test_loader):
    print(f"\n⏳ Evaluating {model_path} on test set...")
    
    # FIX: Correctly load state_dict (no weights_only arg)
    model_instance.load_state_dict(torch.load(model_path))
    model_instance.to(device)
    model_instance.eval()
    
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing"):
            tokens, labels, mask = [b.to(device) for b in batch]
            emissions = model_instance(tokens, mask)
            pred_tags = model_instance.decode(emissions, mask)
            
            for i, sent_preds in enumerate(pred_tags):
                real_len = mask[i].sum().item()
                sent_labels = labels[i][:real_len].cpu().numpy()
                sent_preds = sent_preds[:real_len]
                all_preds.append([ID2LABEL[str(p)] for p in sent_preds])
                all_labels.append([ID2LABEL[str(l)] for l in sent_labels])
                
    report_str = classification_report(all_labels, all_preds, digits=4)
    report_dict = classification_report(all_labels, all_preds, output_dict=True)
    print("✅ Evaluation complete. Results:")
    print(report_str)
    return report_dict, all_labels, all_preds

print("✅ All evaluation functions are defined and fixed!")

In [ ]:
# Create a dictionary to store all results
final_results = {}
raw_predictions = {}

# --- Run Track A (BioBERT) ---
results_A, y_true_A, y_pred_A = evaluate_transformer(trainer_A, tokenized_datasets_A["test"])
final_results["BioBERT (Track A)"] = results_A
raw_predictions["BioBERT (Track A)"] = (y_true_A, y_pred_A)

# --- Run Track B (DeBERTa) ---
results_B, y_true_B, y_pred_B = evaluate_transformer(trainer_B, tokenized_datasets_B["test"])
final_results["DeBERTa (Track B)"] = results_B
raw_predictions["DeBERTa (Track B)"] = (y_true_B, y_pred_B)

# --- Run Track C1 (BioWordVec) ---
results_C1, y_true_C1, y_pred_C1 = evaluate_bilstm("Track_C1_BioWordVec_best.pth", model_c1, test_loader)
final_results["BiLSTM-CRF (BioWordVec)"] = results_C1
raw_predictions["BiLSTM-CRF (BioWordVec)"] = (y_true_C1, y_pred_C1)

# --- Run Track C2 (FastText) ---
results_C2, y_true_C2, y_pred_C2 = evaluate_bilstm("Track_C2_FastText_best.pth", model_c2, test_loader)
final_results["BiLSTM-CRF (FastText)"] = results_C2
raw_predictions["BiLSTM-CRF (FastText)"] = (y_true_C2, y_pred_C2)

print("\n🎉🎉🎉 All models have been evaluated! 🎉🎉🎉")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import time
from statsmodels.stats.contingency_tables import mcnemar
from collections import Counter

# --- 1. Learning Curves (Track C) ---
print("\n--- 📈 1. Learning Curves (BiLSTM-CRF) ---")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot C1 (BioWordVec)
ax1.plot(history_c1['train_loss'], label='Train Loss')
ax1_twin = ax1.twinx()
ax1_twin.plot(history_c1['val_f1'], label='Val F1', color='orange')
ax1.set_title('Track C1: BiLSTM-CRF (BioWordVec)')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1_twin.set_ylabel('F1 Score')
ax1.legend(loc='upper left')
ax1_twin.legend(loc='upper right')

# Plot C2 (FastText)
ax2.plot(history_c2['train_loss'], label='Train Loss')
ax2_twin = ax2.twinx()
ax2_twin.plot(history_c2['val_f1'], label='Val F1', color='orange')
ax2.set_title('Track C2: BiLSTM-CRF (FastText)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2_twin.set_ylabel('F1 Score')
ax2.legend(loc='upper left')
ax2_twin.legend(loc='upper right')

plt.tight_layout()
plt.show()

# --- 2. Final Comparison Table ---
print("\n--- 📊 2. Final Comparison Table ---")
table_data = []
for model_name, res in final_results.items():
    table_data.append({
        "Model": model_name,
        "Overall F1 (Weighted)": res['weighted avg']['f1-score'],
        "Overall Precision": res['weighted avg']['precision'],
        "Overall Recall": res['weighted avg']['recall'],
        "Chemical F1": res['Chemical']['f1-score'],
        "Disease F1": res['Disease']['f1-score']
    })

df = pd.DataFrame(table_data).set_index("Model")
print(df.to_markdown(floatfmt=".4f"))

# --- 3. Inference Speed Benchmark ---
print("\n--- ⏱️ 3. Inference Speed Benchmark ---")
# Transformers (Tracks A & B)
start_A = time.time()
trainer_A.predict(tokenized_datasets_A["test"])
time_A = time.time() - start_A

start_B = time.time()
trainer_B.predict(tokenized_datasets_B["test"])
time_B = time.time() - start_B

# BiLSTM-CRF (Tracks C1 & C2)
@torch.no_grad()
def time_bilstm(model, loader):
    model.eval()
    start = time.time()
    for batch in loader:
        tokens, labels, mask = [b.to(device) for b in batch]
        model(tokens, mask) # Just the forward pass
    return time.time() - start

time_C1 = time_bilstm(model_c1, test_loader)
time_C2 = time_bilstm(model_c2, test_loader)

print(f"BioBERT (Track A) Inference Time: {time_A:.4f} seconds")
print(f"DeBERTa (Track B) Inference Time: {time_B:.4f} seconds")
print(f"BiLSTM-CRF (C1) Inference Time: {time_C1:.4f} seconds")
print(f"BiLSTM-CRF (C2) Inference Time: {time_C2:.4f} seconds")

# --- 4. Statistical Significance (McNemar's Test) ---
print("\n--- 🔬 4. Statistical Significance (Comparing DeBERTa vs. BioBERT) ---")
# This test requires a contingency table of errors
# We need to "flatten" the predictions to a list of tags
y_true_flat_A = [tag for sent in y_true_A for tag in sent]
y_pred_flat_A = [tag for sent in y_pred_A for tag in sent]
y_true_flat_B = [tag for sent in y_true_B for tag in sent]
y_pred_flat_B = [tag for sent in y_pred_B for tag in sent]

# Ensure lists are the same length (they should be)
if len(y_true_flat_A) == len(y_true_flat_B):
    # b = A wrong, B right
    # c = A right, B wrong
    b_count, c_count = 0, 0
    for i in range(len(y_true_flat_A)):
        true = y_true_flat_A[i]
        pred_A = y_pred_flat_A[i]
        pred_B = y_pred_flat_B[i]
        
        if pred_A == true and pred_B != true:
            c_count += 1 # A right, B wrong
        elif pred_A != true and pred_B == true:
            b_count += 1 # A wrong, B right
            
    contingency_table = [[max(1, b_count + c_count), b_count], [c_count, max(1, b_count + c_count)]]
    result = mcnemar(contingency_table, exact=True)
    
    print(f"Model A (BioBERT) right, B (DeBERTa) wrong: {c_count} times")
    print(f"Model A (BioBERT) wrong, B (DeBERTa) right: {b_count} times")
    print(f"p-value: {result.pvalue:.4f}")
    if result.pvalue < 0.05:
        print("Conclusion: The difference is statistically significant. (p < 0.05)")
    else:
        print("Conclusion: The difference is NOT statistically significant. (p >= 0.05)")
else:
    print("Error: Prediction lists have different lengths. Cannot run test.")

# --- 5. Error Analysis (Most Common Mistakes for DeBERTa) ---
print("\n--- 🚫 5. Error Analysis (Top 20 Errors for DeBERTa) ---")
error_counter = Counter()
for sent_true, sent_pred in zip(y_true_B, y_pred_B):
    for tag_true, tag_pred in zip(sent_true, sent_pred):
        if tag_true != tag_pred:
            # We only care about entity errors, not 'O' vs 'O'
            if tag_true != 'O' or tag_pred != 'O':
                error_counter[f"True: {tag_true} | Pred: {tag_pred}"] += 1

print("Most common errors (True Label | Predicted Label):")
for error, count in error_counter.most_common(20):
    print(f"  {count}\t{error}")